# Trabajo Nº2 – Clasificación de Spam y No Spam con NLP

Este notebook sirve como plantilla mínima para que puedas comenzar a construir tus propios modelos de clasificación de texto.  
Incluye:
- Configuración básica e imports.
- Carga de los datasets `train.csv` y `test.csv` desde la carpeta `Spam_Data/`.
- Celda final para generar el archivo `.csv` de predicciones con el formato requerido.

Indicaciones:
- En la entrega se proporciona `Spam_Data.zip`, que contiene:
  - `train.csv` (con la columna `spam_label` como variable objetivo)
  - `test.csv` (sin etiquetas)
  - `sample_submission.csv` (ejemplo del formato del .csv)
- Descomprime el zip en el mismo directorio del notebook antes de ejecutar.
- Tú debes definir, entrenar y comparar tus modelos.
- Al final, genera `submission.csv` con el formato especificado para la entrega.

In [ ]:
# Añade aquí los import que utilices en tu código
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn import metrics
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import text as sk_text

pd.set_option('display.max_rows', 36)
pd.set_option("display.max_colwidth", 150)
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(33)
tf.random.set_seed(33)


## Carga de Datos

Se cargan los conjuntos **train** y **test** desde la carpeta `dataset`, asumiendo esta estructura:

```text
dataset/
├─ train.csv
├─ test.csv
├─ sample_submission.csv
```

Contenido:

- `train.csv` contiene las columnas **`row_id`**, **`text`** y **`spam_label`** (0 = NO SPAM, 1 = SPAM).
- `test.csv` contiene las columnas **`row_id`** y **`text`** (sin etiquetas).
- `sample_submission.csv` muestra el formato de entrega requerido.

Las etiquetas se almacenan como enteros:

- `0` → NO SPAM
- `1` → SPAM

In [ ]:
DATASET_DIR = "dataset"  # Cambia si usas otra ruta

# Cargar los archivos de train
train = pd.read_csv(f"{DATASET_DIR}/train.csv", index_col="row_id")
train.head()


In [ ]:
# Cargar los archivos de test
test = pd.read_csv(f"{DATASET_DIR}/test.csv", index_col="row_id")
test.head()


## Desarrollo de la Práctica

### Exploración breve del conjunto

Seguimos la estructura de los apuntes: antes de vectorizar, revisamos rápidamente la distribución de clases y algunos ejemplos para comprobar la calidad del texto.

In [ ]:
# Distribución de clases
train[['spam_label']].value_counts(normalize=True)

# Ejemplos iniciales
train.sample(5, random_state=33)

### Preprocesamiento ligero y partición

Tomamos el esquema de bag-of-words/TF–IDF de los apuntes. Aplicamos una limpieza mínima: minúsculas, eliminación de caracteres no alfanuméricos y stopwords básicas en inglés. Después hacemos un `train_test_split` para validar el modelo.

In [ ]:
stopwords = sk_text.ENGLISH_STOP_WORDS

def limpiar_texto(texto: str) -> str:
    texto = texto.lower()
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    palabras = [t for t in texto.split() if t not in stopwords]
    return ' '.join(palabras)

train['text_clean'] = train['text'].apply(limpiar_texto)

X_train, X_val, y_train, y_val = train_test_split(
    train['text_clean'], train['spam_label'], test_size=0.2, random_state=33, stratify=train['spam_label']
)
print(f'Tamaño train: {X_train.shape[0]}, validación: {X_val.shape[0]}')

### Vectorización TF–IDF y modelo lineal

Se utiliza un pipeline sencillo `TFIDF + LogisticRegression`, una combinación clásica que se cubre en los apuntes. Mantenemos hiperparámetros modestos para no complicar la práctica.

In [ ]:
pipeline = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(max_features=15000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=200, n_jobs=-1, class_weight='balanced', random_state=33)),
])

pipeline.fit(X_train, y_train)

y_val_pred = pipeline.predict(X_val)

resultados = {
    'accuracy': metrics.accuracy_score(y_val, y_val_pred),
    'precision': metrics.precision_score(y_val, y_val_pred),
    'recall': metrics.recall_score(y_val, y_val_pred),
    'f1': metrics.f1_score(y_val, y_val_pred),
    'mcc': matthews_corrcoef(y_val, y_val_pred),
}
resultados

### Entrenamiento final y predicción sobre el test

Con el mismo pipeline ajustamos sobre todo el conjunto de entrenamiento (ya preprocesado) y generamos `y_pred` para reutilizar la celda de exportación de la plantilla.

### Modelo alternativo: Bolsa de Palabras + Naive Bayes

Basado en los apuntes de BoW/TF–IDF, añadimos un modelo clásico de **Bolsa de Palabras** con suavizado de Laplace y predicción Naive Bayes Multinomial. Se mantiene el mismo preprocesado y partición estratificada para comparar métricas con el pipeline lineal.

In [ ]:
pipeline_nb = Pipeline(steps=[
    ('bow', CountVectorizer(max_features=15000, ngram_range=(1, 2), stop_words=stopwords)),
    ('clf', MultinomialNB(alpha=1.0)),
])

pipeline_nb.fit(X_train, y_train)

y_val_nb_pred = pipeline_nb.predict(X_val)

resultados_nb = {
    'accuracy': metrics.accuracy_score(y_val, y_val_nb_pred),
    'precision': metrics.precision_score(y_val, y_val_nb_pred),
    'recall': metrics.recall_score(y_val, y_val_nb_pred),
    'f1': metrics.f1_score(y_val, y_val_nb_pred),
    'mcc': matthews_corrcoef(y_val, y_val_nb_pred),
}
resultados_nb

### Entrenamiento final del modelo Naive Bayes

Ajustamos el modelo de Bolsa de Palabras sobre todo el conjunto de entrenamiento y generamos predicciones sobre el test para poder comparar distribuciones con el pipeline TF–IDF + Regresión Logística.

In [ ]:
pipeline_nb_final = pipeline_nb
pipeline_nb_final.fit(train['text_clean'], train['spam_label'])

test['text_clean'] = test['text'].apply(limpiar_texto)

y_pred_nb = pipeline_nb_final.predict(test['text_clean'])

unique_nb, counts_nb = np.unique(y_pred_nb, return_counts=True)
print(dict(zip(unique_nb, counts_nb)))

In [ ]:
pipeline_final = pipeline
pipeline_final.fit(train['text_clean'], train['spam_label'])

test['text_clean'] = test['text'].apply(limpiar_texto)

y_pred = pipeline_final.predict(test['text_clean'])

unique, counts = np.unique(y_pred, return_counts=True)
print(dict(zip(unique, counts)))

## Generación del CSV final de Predicciones

Una vez **definido y entrenado tu modelo** (no incluido en esta plantilla) y generadas tus predicciones sobre el conjunto de prueba `test`, crea el archivo `submission.csv` con el siguiente formato:

```text
row_id,spam_label
7097,0
7098,0
7099,1
...
````

Donde:

* `row_id` es el identificador del texto en el conjunto de test.
* `spam_label` es `0` para **NO SPAM** y `1` para **SPAM**.

In [ ]:
# Cargar el archivo de ejemplo de envío
submission = pd.read_csv(f"{DATASET_DIR}/sample_submission.csv")

# Reemplazar la columna con tus predicciones
# (asegúrate de que 'y_pred' contenga las predicciones finales del modelo)
submission["spam_label"] = y_pred

# Guardar el archivo en el formato requerido
output_path = f"{DATASET_DIR}/submission.csv"
submission.to_csv(output_path, index=False)

print(f"Archivo guardado correctamente como: {output_path}")
submission.head()